# 05 — Recurring estimator stress calibration
This job monitors single-estimator behavior across random historical pseudo-assignments.
It is useful regression evidence, but it does **not** authorize the selected multi-
estimator design in notebook 03 because it does not replay that selector or decision rule.

In [ ]:
%pip install /Workspace/Shared/supply_experiments/supply_experiments-2.0.0a1-py3-none-any.whl --quiet --force-reinstall --no-deps
dbutils.library.restartPython()

In [ ]:
from supply_experiments.io import load_city_panel, TABLES
from supply_experiments.calibration.aa import run_aa_calibration
from supply_experiments.estimators.scm import fit_scm
from supply_experiments.estimators.ascm import fit_ascm
from supply_experiments.estimators.sdid import fit_sdid
import supply_experiments

tables = TABLES["sandbox"]
panel, stats = load_city_panel(spark, "2025-06-01", "2026-06-30")
eligible = [c for c in stats.index if c in panel.cities][:60]

In [ ]:
from datetime import datetime
rows = []
for name, fn in [("scm", fit_scm), ("ascm", fit_ascm), ("sdid", fit_sdid)]:
    aa = run_aa_calibration(panel, eligible, fn, pre_days=120, post_days=35,
                            n_runs=200, n_treated=2, n_donors=15, alpha=0.10)
    print(f"{name}: FPR={aa.fpr:.1%}, upper95={aa.fpr_upper_bound:.1%}, "
          f"invalid={aa.invalid_rate:.1%}, rank-PIT KS={aa.ks_p_value:.3f}, "
          f"passed={aa.passed}")
    rows.append({"run_at": datetime.utcnow(), "estimator": name,
                 "package_version": supply_experiments.__version__,
                 "n_runs": aa.n_runs, "alpha": aa.alpha, "fpr": aa.fpr,
                 "fpr_ci_low": aa.fpr_ci[0], "fpr_ci_high": aa.fpr_ci[1],
                 "ks_p_value": aa.ks_p_value, "median_bias": aa.median_att_bias,
                 "invalid_rate": aa.invalid_rate,
                 "calibration_fingerprint": aa.calibration_fingerprint,
                 "design_fingerprint": aa.design_fingerprint,
                 "selection_scope": aa.selection_scope,
                 "procedure_scope": aa.procedure_scope,
                 "artifact_json": aa.to_json(include_details=False),
                 "passed": aa.passed})
spark.createDataFrame(rows).write.format("delta").mode("append") \
    .option("mergeSchema", "true").saveAsTable(tables["calibration"])